In [13]:
import numpy as np
import scipy.fftpack
from scipy.fftpack import dct
from sklearn.decomposition import IncrementalPCA, PCA
import pickle as pkl
import librosa
import os
import matplotlib.pyplot as plt
from scipy.signal import lfilter
plt.close('all')

In [14]:
def create_custom_mel_filterbank(n_fft, sr, filter_params):
    """
    Cria um banco de filtros mel personalizados.
    
    Args:
        n_fft (int): Tamanho da Transformada de Fourier Rápida (FFT).
        sr (int): Taxa de amostragem do sinal de áudio.
        filter_params (list): Lista de tuplas definindo frequências centrais
                              e larguras de banda para cada filtro mel.
    
    Returns:
        numpy.ndarray: Matriz representando o banco de filtros mel.
    """
    n_filters = len(filter_params)
    filterbank = np.zeros((n_filters, int(np.floor(n_fft / 2 + 1))))
    
    for m, (freq_center, bandwidth) in enumerate(filter_params):
        # Calcula frequências centrais e larguras de banda em escala mel
        mel_center = 2595 * np.log10(1 + freq_center / 700)
        mel_bandwidth = 2595 * np.log10(1 + (freq_center + bandwidth) / 700) - mel_center
        
        # Converte frequências mel para escala linear
        hz_center = 700 * (10 ** (mel_center / 2595) - 1)
        hz_bandwidth = 700 * (10 ** (mel_bandwidth / 2595) - 1)
        
        # Calcula índices dos bins do filtro na FFT
        bin_idxs = np.floor(
            (n_fft + 1) * np.array([
                hz_center - hz_bandwidth / 2,
                hz_center,
                hz_center + hz_bandwidth / 2
            ]) / sr
        ).astype(int)
        
        # Preenche os pesos do filtro de acordo com a distância dos bins
        for k in range(bin_idxs[0], bin_idxs[2] + 1):
            if k <= bin_idxs[1]:
                filterbank[m, k] = (k - bin_idxs[0]) / (bin_idxs[1] - bin_idxs[0])
            else:
                filterbank[m, k] = (bin_idxs[2] - k) / (bin_idxs[2] - bin_idxs[1])
    
    return filterbank


# Definição dos parâmetros dos filtros MEL (Largura de Banda e Frequência Central)
# especificado na TABELA 3.1
filter_params = [
    (20, 50),      # Filtro 1: Frequência central = 20 Hz, Largura de banda = 50 Hz
    # ... (outros filtros omitidos)
    (6500, 1000),  # Filtro 42: Frequência central = 6500 Hz, Largura de banda = 1000 Hz
]

In [15]:
### PROCESSAMENTO DAS PASTAS CONTENDO OS ARQUIVOS .wav ######################
"""
Um diretório principal é composto por pastas contendo os arquivos .wav de cada
classe (uma pasta para cada classe).
Além disso, esse diretório também contém uma pasta com os códigos e outra pasta
com o resultado da simulação (matriz de vetores de características para cada classe).
O código itera por cada pasta no diretório principal, extraindo características
dos arquivos de áudio (.wav) e salvando os resultados em arquivos pickle.
"""

import os
import numpy as np
import librosa
from scipy.signal import lfilter
from scipy.fftpack import dct

# ------------------------------------------------------------------ #
# Importe aqui a função create_custom_mel_filterbank e a lista
# filter_params definidas anteriormente (ou coloque-as neste mesmo arquivo)
# ------------------------------------------------------------------ #

diretorio = "L:/1-TESE MESTRADO/EXPERIMENTO2"  # diretório principal onde estão armazenadas
                                                # as pastas contendo os arquivos de áudio (.wav)
pastas = os.listdir(diretorio)
pastas = [pasta for pasta in pastas if pasta not in ['simulacao', 'codigos']]

a = 0  # Iniciar o vetor de rotulagem de classes
b = 0
intervalo_extracao = 4  # Duração do intervalo para extração das características (segundos)

for pasta in pastas:
    print(f"Processando pasta {pasta}")
    os.chdir(os.path.join(diretorio, pasta))  # Mudar para o diretório da pasta atual
    arquivos = os.listdir()                   # Lista todos os arquivos no diretório atual
    nome_pasta = str(pasta)
    dados = []

    # Filtrar apenas arquivos .wav
    for arquivo in arquivos:
        if arquivo.endswith(".wav") or arquivo.endswith(".WAV"):
            dados.append(arquivo)

    features_da_class = []  # Características de todos os arquivos da pasta
    nomes_arquivos = []

    for file in dados:
        # Carregar o arquivo de áudio
        X, sr = librosa.load(file, sr=50000, mono=False)

        # Segmentação do arquivo e extração de features (MFCC, SDF, LPC, PSC, etc.)
        n_amostras_intervalo = int(intervalo_extracao * sr)  # Tamanho do segmento
        total_amostras = np.size(X)                         # Duração do sinal (em amostras)
        num_segmentos = int(np.floor(total_amostras / n_amostras_intervalo)) \
            if n_amostras_intervalo <= total_amostras else 1

        features_do_arquivo = []
        arquivo_nome = os.path.splitext(file)[0]  # Nome do arquivo sem extensão

        for j in range(num_segmentos):
            # Define o intervalo do sinal para análise no segmento atual
            amostra_inicial = n_amostras_intervalo * j
            amostra_final = amostra_inicial + n_amostras_intervalo
            signal = X[amostra_inicial:amostra_final]

            # ---------------- Extração dos vetores de características ----------------
            # LPC e LPCC
            lpc = librosa.lpc(signal, order=12, axis=0)  # Coeficientes LPC

            def lpc_to_lpcc(lpc, order_lpcc):
                return -lfilter([1] + lpc[1:], [1], np.arange(1, order_lpcc + 1))

            lpcc = lpc_to_lpcc(lpc, 12)  # LPCCs

            # MFCCs tradicionais
            mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=42).T
            mean_mfccs = np.mean(mfccs, axis=0)  # Média dos MFCCs

            # PSC (Power Spectrum Coefficient)
            stft = np.abs(np.fft.fft(signal))
            power_spectrum = np.abs(stft) ** 2
            PSC = np.mean(power_spectrum, axis=0)

            # ZCR (Zero Crossing Rate)
            zcr = librosa.feature.zero_crossing_rate(signal)
            zcr = zcr.reshape(-1)

            # Pré-ênfase para realçar altas frequências
            pre_emphasis = 0.97
            emphasized_signal = np.append(
                signal[0], signal[1:] - pre_emphasis * signal[:-1]
            )

            # Enquadramento do sinal em frames com salto (overlap)
            frame_size = 40e-3               # Tamanho do frame (segundos)
            hop = (1 / 2) * (40e-3)          # Salto entre frames (overlap 50%)
            frame_length, hop_length = frame_size * sr, hop * sr
            signal_length = len(emphasized_signal)

            frame_length = int(round(frame_length))
            hop_length = int(round(hop_length))

            num_frames = int(np.ceil(float(np.abs(signal_length - frame_length)) / hop_length))
            pad_signal_length = num_frames * hop_length + frame_length
            z = np.zeros((pad_signal_length - signal_length))
            pad_signal = np.append(emphasized_signal, z)  # Acoplamento de zeros

            # Criação de índices para extração de frames
            indices = np.tile(np.arange(0, frame_length), (num_frames, 1)) + np.tile(
                np.arange(0, num_frames * hop_length, hop_length), (frame_length, 1)
            ).T
            frames = pad_signal[indices.astype(np.int32, copy=False)]

            # Janelamento (função Hamming) para reduzir vazamento espectral
            frames *= np.hamming(frame_length)

            # FFT
            NFFT = frame_length
            mag_frames = np.abs(np.fft.rfft(frames, NFFT))   # Magnitude do espectro
            pow_frames = ((1.0 / NFFT) * ((mag_frames) ** 2))  # Espectro de potência

            # ---------------- Banco de filtros Mel projetado ----------------
            filterbank = create_custom_mel_filterbank(NFFT, sr, filter_params)

            # Aplicação dos filtros Mel
            filter_banks = np.dot(pow_frames, filterbank.T)
            filter_banks = np.where(
                filter_banks == 0, np.finfo(float).eps, filter_banks
            )  # Estabilidade numérica
            filter_output = filter_banks  # Saída do filtro

            # Conversão do espectro de potência em dB
            filter_banks = 20 * np.log10(filter_banks)  # dB

            # Cálculo de MFCCs utilizando o banco de filtros projetado
            num_ceps = len(filter_params)
            mfcc = dct(filter_banks, type=2, axis=1, norm='ortho')[:, 0:num_ceps]
            MFCC = np.mean(mfcc, axis=0)  # vetor MFCC para o banco de filtros projetado

FileNotFoundError: [WinError 3] O sistema não pode encontrar o caminho especificado: 'L:/1-TESE MESTRADO/EXPERIMENTO2'

In [16]:
#################################################################################
##### Modificação para extrair features dinâmicas #####
#################################################################################

sdf = np.zeros((np.size(filter_output, axis=1), np.size(filter_output, axis=0)))

for ii in range(np.size(filter_output, axis=1)):
    y = filter_output[:, ii]                    # Coluna do espectro de potência
    fft = np.abs(scipy.fftpack.fft(y))          # FFT da coluna
    pow_y = ((1.0 / len(y)) * ((fft) ** 2))     # Espectro de potência da coluna
    sdf[ii, :] = fft

sdf = 20 * np.log10(sdf)                        # Conversão para dB

n_p = 4  # Número de pontos para DCT
sdf = dct(sdf, type=2, axis=1, norm='ortho')[:, 0:(n_p)]  # DCT do espectro dinâmico

n_point = 4
filter_output = dct(filter_output, type=2, axis=1, norm='ortho')[:, 1:(n_point + 1)]

SDF = sdf.flatten()  # Vetor de recursos espectrais dinâmicos (Vetor SDF)

# Combinação dos vetores de características
"""
A próxima linha define qual vetor de características ou combinação deles será
extraído e colocado em uma matriz que posteriormente será submetida ao classificador.
"""
ext_features = np.hstack([MFCC, SDF])  # Vetor de característica que será submetido ao classificador

features_do_arquivo.append(ext_features)
features_do_arquivo = np.c_[features_do_arquivo]  # Concatenação dos vetores de características de todos os segmentos

new_column = np.array([arquivo_nome] * features_do_arquivo.shape[0])  # Array NumPy com o nome do arquivo repetido
features_do_arquivo = np.concatenate(
    (features_do_arquivo, new_column[:, np.newaxis]), axis=1
)  # Adiciona nome do arquivo como última coluna

features_da_class.append(features_do_arquivo)
features_da_class = np.vstack(features_da_class)  # Concatena os vetores de características de todos os arquivos da pasta

# Gera o vetor de classe
size = features_da_class.shape[0]
classe = np.zeros(size)

for i in np.arange(size):
    classe[i] = a   # Define a classe (valor atribuído manualmente)
    a = a + 1       # Incrementa o valor da classe para a próxima pasta

# Concatena features e classe, inverte a ordem das duas últimas colunas
resultado = np.c_[features_da_class, classe]
resultado[:, -2], resultado[:, -1] = resultado[:, -1], resultado[:, -2].copy()

# Salva os dados em um arquivo pickle
os.chdir("L:\\1-TESE MESTRADO\\EXPERIMENTO2\\simulacao")
with open(str(pasta) + '.pkl', 'wb') as i:
    pkl.dump(resultado, i)

NameError: name 'filter_output' is not defined